# Neural covariate effects

The Cox model of notebook 01 assumes **proportional hazards with a linear
predictor**: the log-hazard is $x\cdot\beta$, a weighted sum of the covariates.
That is often too rigid. Real risk can be **nonlinear** in a covariate, or depend
on an **interaction** between covariates (risk rises only when two factors are high
together). A linear predictor represents neither.

The fix keeps the same TPP model and changes one thing: **where the parameters come
from**. In notebook 03 a plain `nn.Linear` mapped the covariates to the event-time
density's parameters. Here we swap that for a small **MLP** (a neural network with
one hidden layer). The model class, the hooks, the fit loop, and the survival
contract are all unchanged — only the parameter map gains a hidden layer, and with
it the ability to fit nonlinear and interacting covariate effects.

We compare against a linear Cox baseline on data whose true risk is nonlinear, and
show the MLP wins on held-out log-likelihood, on concordance (ranking who fails
first), and in recovering the true risk shape.

In [ ]:
import math

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torchpp import BatchedMVEventData, DensityBasedTPP, IntensityBasedTPP, TokenRoles
from torchpp.modules.nn.lognormal_mixture import LogNormalMixtureDistribution
from survival_tools import (
    make_nonlinear_covariate,
    survival_batch, covariate_batch, fit_full_batch,
    model_survival_curve, concordance_index,
    plot_calibration,
)

# survival_tools selects the headless Agg backend on import; switch to inline
# afterwards so figures render in the notebook.
%matplotlib inline
_ = torch.manual_seed(0)

## 1. Data with a nonlinear, interacting risk

`make_nonlinear_covariate` draws an event time with a constant baseline hazard
$h_0$ and a **nonlinear log-risk** $g(x)$: the hazard is $h_0\,e^{g(x)}$. We choose

$$g(x) = 0.8\,x_0 \;+\; 1.0\,x_1 x_2 \;+\; 0.6\,\sin(\pi x_3).$$

Only the first term is linear. The second is an **interaction** ($x_1 x_2$: risk
depends on the product), the third a **nonlinearity** ($\sin$). A linear Cox model
can fit the $x_0$ term but is blind to the other two. We hold out 30% of subjects
for validation.

In [ ]:
def g_effect(x):
    # true log-risk: one linear term, one interaction, one nonlinearity
    return 0.8 * x[:, 0] + 1.0 * x[:, 1] * x[:, 2] + 0.6 * torch.sin(math.pi * x[:, 3])


seqs, meta = make_nonlinear_covariate(
    n=3000, n_features=4, effect_fn=g_effect, h0=1.0, T_obs=3.0, seed=1
)
print(f"subjects: {len(seqs)}    censoring rate: {1 - meta['observed'].float().mean():.1%}")

# 70 / 30 train / validation split
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(seqs), generator=g).tolist()
cut = int(0.7 * len(seqs))
train_idx, val_idx = perm[:cut], perm[cut:]
train_batch = survival_batch([seqs[i] for i in train_idx])
val_batch = survival_batch([seqs[i] for i in val_idx])

# validation truth, and an empty-history batch for predictive queries
val_x = meta["x"][val_idx]
val_event_time = meta["event_time"][val_idx]
val_observed = meta["observed"][val_idx]
val_cov = covariate_batch(val_x, T_obs=3.0)

## 2. Baseline: a linear Cox model

The constant-hazard Cox model from notebook 01, with a log-hazard linear in the
covariates, $x\cdot\beta$. We fit it and record two held-out numbers:

- the **log-likelihood per event**;
- the **concordance index** (Harrell's C) — the fraction of subject pairs whose
  predicted risk ranks them in the order they actually failed ($C=0.5$ is random,
  $C=1$ perfect). For a constant-hazard model the risk score is the rate
  $h_0\,e^{x\cdot\beta}$: higher rate means a sooner event.

In [ ]:
class CoxTPP(IntensityBasedTPP):
    '''Constant-hazard proportional-hazards model with a linear predictor (notebook 01).'''

    def __init__(self, n_features):
        super().__init__(D=1, terminating=True, token_roles=TokenRoles(has_origin=True))
        self.log_h0 = nn.Parameter(torch.zeros(()))       # log baseline hazard
        self.beta = nn.Parameter(torch.zeros(n_features))

    def _rate(self, ts):
        return torch.exp(self.log_h0) * torch.exp(ts.covariates["x"] @ self.beta)  # (B,)

    def _intensities_at_queries(self, query_times, ts):
        B, Q = query_times.shape
        return self._rate(ts).view(B, 1, 1).expand(B, Q, self.D)

    def _closed_form_ci(self, ts, T_low, T):
        hi = self._effective_T(T, ts)
        lo = self._effective_T(T_low, ts) if T_low is not None else 0.0
        return self._rate(ts).unsqueeze(-1) * (hi - lo)


torch.manual_seed(0)
cox_model = CoxTPP(n_features=4)
cox_result = fit_full_batch(cox_model, train_batch, steps=800, lr=0.05, val_batch=val_batch, log_every=0)

cox_risk = cox_model._rate(val_cov).detach().numpy()          # higher = fails sooner
cox_C = concordance_index(cox_risk, val_event_time.numpy(), val_observed.numpy())
print(f"linear Cox    val log-lik/event: {cox_result['val_loglik']:.3f}    concordance: {cox_C:.3f}")

## 3. A neural density model

The same `DensityBasedTPP` survival model as notebook 03 — a log-normal mixture on
the event time — but the parameter map is now a small MLP
(`Linear → Tanh → Linear`) instead of a single `Linear`. The MLP reads the four
covariates and produces the mixture parameters, so the event-time distribution can
bend with the covariates in a nonlinear, interacting way.

We keep the mixture small ($K=3$) so the extra flexibility lives in the covariate
map, not in the time shape — the win should come from modelling $g(x)$ better, not
from a richer density. (A large mixture on top of the MLP overfits: each subject
has a single event time, which a very flexible conditional density can memorise.)

The risk score for concordance is $-E[t_\text{next}]$: a shorter expected event
time means a higher risk.

In [ ]:
from torchpp.utils.event_data import scored_event_mask

# Numerical guards, mirroring the library's intensity-free model.
_EPS = 1e-8            # floor on the inter-event time (log-normal support is t > 0)
_LOG_SCALE_MIN = -2.0  # floor on the log-scale: a tiny scale gives huge log-density gradients
_LOG_SCALE_MAX = 3.0


class LogNormalMixtureSurvival(DensityBasedTPP):
    """Survival model: a covariate-conditioned log-normal mixture on the event time.

    Identical to notebook 03; only `param_net` differs (here an MLP).  D=1 (one
    event type), terminating (fires at most once), has_origin (the event is the
    scored outcome).  `param_net` maps the covariates x to the 3K mixture
    parameters [K locations, K log-scales, K log-weights].
    """

    def __init__(self, param_net, num_mix_components=8):
        super().__init__(D=1, terminating=True, token_roles=TokenRoles(has_origin=True))
        self.K = int(num_mix_components)
        self.param_net = param_net

    def _dist(self, raw):
        # Split the raw params into a log-normal mixture over the inter-event time.
        K = self.K
        locs = raw[..., :K]
        log_scales = raw[..., K:2 * K].clamp(_LOG_SCALE_MIN, _LOG_SCALE_MAX)
        log_weights = torch.log_softmax(raw[..., 2 * K:3 * K], dim=-1)
        return LogNormalMixtureDistribution(locs=locs, log_scales=log_scales, log_weights=log_weights)

    def _dist_and_tau(self, t, ts):
        # Per-subject mixture, and the inter-event time tau = t - t_last.
        raw = self.param_net(ts.covariates["x"])          # (B, 3K)
        t_last = self._last_event_time(ts)                # (B,) = 0 for an empty history
        if t.dim() == 2:                                  # (B, Q) query grid
            raw = raw.unsqueeze(1).expand(raw.shape[0], t.shape[1], raw.shape[-1])
            tau = (t - t_last.unsqueeze(1)).clamp(min=_EPS)
        else:
            tau = (t - t_last).clamp(min=_EPS)
        return self._dist(raw.to(t.dtype)), tau

    # --- density hooks ------------------------------------------------------
    def _log_density_time_given_mark(self, t, ts):
        if t.shape[0] == 0:
            return torch.zeros(*t.shape, self.D, dtype=t.dtype, device=t.device)
        dist, tau = self._dist_and_tau(t, ts)
        log_f = dist.log_prob(tau)                        # (B,) or (B, Q)
        return log_f.unsqueeze(-1).expand(*log_f.shape, self.D)   # broadcast over D=1

    def _log_survival_time_given_mark(self, t, ts):
        if t.shape[0] == 0:
            return torch.zeros(*t.shape, self.D, dtype=t.dtype, device=t.device)
        dist, tau = self._dist_and_tau(t, ts)
        log_S = dist.log_survival_function(tau)
        # A subject whose event has fired has no next event: log S = 0, so its
        # censoring term drops and the likelihood keeps only the event density.
        fired = ~self._get_termination_mask(ts).squeeze(-1)      # (B,)
        if log_S.dim() == 2:
            fired = fired.unsqueeze(1)
        log_S = log_S.masked_fill(fired, 0.0)
        return log_S.unsqueeze(-1).expand(*log_S.shape, self.D)

    def _log_prob_mark(self, ts):
        # One event type -> log p(m) = log 1 = 0.
        B = ts.time_points.shape[0]
        return torch.zeros(B, self.D, dtype=ts.time_points.dtype, device=ts.time_points.device)

    def _event_log_density_sum(self, ts):
        # Sum the observed events' log-density, over the scored events only.
        B, L = ts.time_points.shape
        dtype, device = ts.time_points.dtype, ts.time_points.device
        if L == 0:
            return torch.zeros(B, dtype=dtype, device=device)
        scored = scored_event_mask(ts, self.predictable_mask, self.has_origin)                    # (B, L)
        raw = self.param_net(ts.covariates["x"]).unsqueeze(1).expand(B, L, 3 * self.K)
        dist = self._dist(raw.to(dtype))
        prev = torch.cat([torch.zeros(B, 1, dtype=dtype, device=device), ts.time_points[:, :-1]], dim=1)
        tau = (ts.time_points - prev).clamp(min=_EPS)     # time since the previous event
        return dist.log_prob(tau).masked_fill(~scored, 0.0).sum(dim=1)

    # --- sampling + point prediction ---------------------------------------
    def _draw_next(self, ts_current, rng):
        batched = BatchedMVEventData.from_sequences([ts_current])   # keep the covariates
        dist = self._dist(self.param_net(batched.covariates["x"]))
        gmm = dist.base_dist
        k = torch.multinomial(torch.softmax(gmm.mixture_distribution.logits, -1), 1, generator=rng)
        loc = gmm.component_distribution.loc.gather(-1, k).squeeze(-1)
        scale = gmm.component_distribution.scale.gather(-1, k).squeeze(-1)
        tau = (loc + scale * torch.randn(loc.shape, generator=rng, device=loc.device)).exp()  # log-normal draw
        t_next = self._last_event_time(batched) + tau
        return t_next, torch.zeros(1, dtype=torch.long), torch.ones(1)

    def time_mean(self, ts):
        # E[t_next] = t_last + sum_k w_k exp(loc_k + 0.5 scale_k^2)  (log-normal mixture mean).
        K = self.K
        raw = self.param_net(ts.covariates["x"])
        locs = raw[..., :K]
        scales = raw[..., K:2 * K].clamp(_LOG_SCALE_MIN, _LOG_SCALE_MAX).exp()
        log_w = torch.log_softmax(raw[..., 2 * K:3 * K], dim=-1)
        e_tau = torch.logsumexp(log_w + locs + 0.5 * scales ** 2, dim=-1).exp()
        return self._last_event_time(ts) + e_tau

In [ ]:
torch.manual_seed(0)
mlp = nn.Sequential(nn.Linear(4, 16), nn.Tanh(), nn.Linear(16, 3 * 3))
density_model = LogNormalMixtureSurvival(mlp, num_mix_components=3)
mlp_result = fit_full_batch(density_model, train_batch, steps=800, lr=0.02, val_batch=val_batch)

mlp_risk = (-density_model.time_mean(val_cov)).detach().numpy()   # higher = shorter expected time
mlp_C = concordance_index(mlp_risk, val_event_time.numpy(), val_observed.numpy())
print(f"\nMLP density   val log-lik/event: {mlp_result['val_loglik']:.3f}    concordance: {mlp_C:.3f}")

## 4. The MLP wins where the covariate effect is nonlinear

Both held-out numbers favour the MLP: a higher log-likelihood (it fits the event
times better) and a higher concordance (it ranks who fails first better). The gap
is the nonlinear risk the linear model cannot see.

We also check **calibration** at a fixed horizon $t$: bin subjects by their
predicted survival $S(t\mid x)$, then compare each bin's mean prediction to the
observed fraction still event-free at $t$. A well-calibrated model sits on the
diagonal.

In [ ]:
print(f"linear Cox    log-lik {cox_result['val_loglik']:.3f}    C {cox_C:.3f}")
print(f"MLP density   log-lik {mlp_result['val_loglik']:.3f}    C {mlp_C:.3f}")

horizon = 1.0
with torch.no_grad():
    S_at_h = model_survival_curve(density_model, val_x, torch.tensor([horizon]))[:, 0].numpy()
survived_past_h = (val_event_time.numpy() > horizon).astype(float)

fig = plot_calibration(S_at_h, survived_past_h, t=horizon)
fig.axes[0].set_title(f"MLP density-model calibration at t = {horizon}")
plt.show()

## 5. Recovering the risk shape

To see *what* the MLP learned, sweep one covariate and read off each model's implied
risk. We vary $x_3$ (the $\sin$ term), hold the other covariates at 0, and plot the
predicted log-risk against the true $g(x)$. For an empty-history subject the
expected event time falls as the hazard rises, so the MLP's risk score is
$-\log E[t_\text{next}]$; the linear Cox uses its log-rate
$\log h_0 + x\cdot\beta$. Curves are centred (mean removed) to compare shapes.

The MLP follows the $\sin$ shape over most of the range (it underestimates the peak
on the left, where data is sparse); the linear Cox, forced to a straight line, misses
it entirely.

In [ ]:
xs = torch.linspace(-2.0, 2.0, 41)
sweep = torch.zeros(41, 4)
sweep[:, 3] = xs
g_true = g_effect(sweep)
sweep_cov = covariate_batch(sweep, T_obs=3.0)

with torch.no_grad():
    mlp_logrisk = -torch.log(density_model.time_mean(sweep_cov))
    cox_logrisk = torch.log(cox_model._rate(sweep_cov))

center = lambda z: (z - z.mean()).numpy()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs.numpy(), center(g_true), "k--", label="true log-risk g(x_3)")
ax.plot(xs.numpy(), center(mlp_logrisk), color="C0", label="MLP density")
ax.plot(xs.numpy(), center(cox_logrisk), color="C3", label="linear Cox")
ax.set_xlabel("x_3")
ax.set_ylabel("log-risk (centred)")
ax.set_title("Learned covariate effect along x_3")
ax.legend(fontsize=8)
plt.show()

print(f"corr(MLP risk, true g) = {np.corrcoef(center(mlp_logrisk), center(g_true))[0, 1]:.3f}")
print(f"corr(Cox risk, true g) = {np.corrcoef(center(cox_logrisk), center(g_true))[0, 1]:.3f}")

## Takeaways

- **Same contract, richer parameter map.** The survival model is exactly the
  `DensityBasedTPP` from notebook 03. Swapping the parameter map from `nn.Linear` to
  a small MLP is the only change — the hooks, the likelihood, and the fit loop are
  untouched.
- **Nonlinear and interacting effects.** A linear Cox predictor cannot represent an
  interaction ($x_1 x_2$) or a nonlinearity ($\sin$). The MLP can, and it beats the
  linear baseline on held-out log-likelihood and concordance and recovers the true
  risk shape along a covariate sweep.
- **Keep the flexibility where it belongs.** A small mixture ($K=3$) puts the gain in
  the covariate map, not in over-flexible time shapes; a large mixture with an MLP
  overfits the one-event-per-subject data.
- **The same trick fits the intensity route.** An `IntensityBasedTPP` can take its
  per-subject hazard (e.g. spline heights via `LinearSpline.interpolate_flat` /
  `integrate_flat`) from an MLP of the covariates in the same way — the parameter
  source is independent of which family computes the likelihood.